**INSTALAÇÃO** **DAS** **BIBLIOTECAS**

In [13]:
!pip install google-generativeai pandas -q

**IMPORTAÇÃO DAS BIBLIOTECAS**

In [14]:
import os
import pandas as pd
import google.generativeai as genai

**CONFIG DE API**

In [15]:
GEMINI_API_KEY = ""

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-flash-latest")

print("Gemini configurado!")

Gemini configurado!


In [16]:
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

 **DEF EXTRAÇÃO DE DADOS**

In [17]:
def load_data():

    jobs = pd.read_csv("/content/jobs.csv")

    users = pd.read_csv("/content/users.csv")

    return jobs, users

**DEF FILTRO DE VAGAS**

In [18]:
def filter_jobs(jobs):

    jobs = jobs[
        jobs['category'].str.contains("Data", na=False)
    ]

    return jobs

**DEF SCORE DE VAGAS**

In [19]:
def score_job(description, skills):

    description = description.lower()

    skills_list = [
        skill.strip()
        for skill in skills.split(",")
    ]

    score = sum(
        skill in description
        for skill in skills_list
    )

    return score

**DEF TOP VAGAS**

In [20]:
def get_top_jobs(jobs, user_skills):

    jobs['score'] = jobs['description'].apply(
        lambda x: score_job(x, user_skills)
    )

    jobs = jobs.sort_values(
        by='score',
        ascending=False
    )

    return jobs.head(3)

DEF TRANSFORMANDO OS DADOS COM IA

In [21]:
def generate_ai_message(user, jobs_df):

    jobs_text = ""

    for _, row in jobs_df.iterrows():

        jobs_text += f"""

        Vaga: {row['title']}
        Empresa: {row['company_name']}
        Compatibilidade: {row['score']}

        """

    prompt = f"""

    Você é um mentor profissional da área de dados.

    Analise o perfil abaixo:

    Nome: {user['name']}
    Skills: {user['skills']}
    Objetivo: {user['goal']}

    Vagas recomendadas:
    {jobs_text}

    Gere uma mensagem personalizada em português contendo:

    - análise do perfil
    - pontos fortes
    - porque as vagas combinam com ele
    - sugestões de melhoria
    - tom profissional e motivador

    """

    response = model.generate_content(prompt)

    return response.text

EXECUTAR PIPELINE PRINCIPAL

In [22]:
jobs, users = load_data()

jobs = filter_jobs(jobs)

print("\nPIPELINE INICIADO...\n")


PIPELINE INICIADO...



LOOP DE MENSAGEM INDIVIDUALIZADO PARA CADA USUARIO

In [23]:
import os

for _, user in users.iterrows():

    top_jobs = get_top_jobs(
        jobs.copy(),
        user['skills']
    )

    message = generate_ai_message(
        user,
        top_jobs
    )

    file_path = f"""
    output/messages/{user['name'].lower()}.txt
    """

    file_path = file_path.strip()

    # Ensure the directory exists
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    with open(
        file_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(message)

    print(f"\nMensagem criada para {user['name']}")
    print("\n============================")
    print(message)
    print("============================\n")


Mensagem criada para Yago

Olá, Yago! É um prazer conversar com você. Como seu mentor na área de dados, analisei cuidadosamente seu perfil e as oportunidades que o mercado está apresentando para o seu conjunto de habilidades.

Você está no caminho certo e possui a base técnica que as empresas mais buscam hoje. Abaixo, apresento uma análise detalhada para te ajudar a conquistar sua vaga:

### 📊 Análise do Perfil
Seu perfil é extremamente estratégico. Você domina a "trindade dourada" da análise de dados: **SQL** para manipulação de bancos de dados, **Python** para automação e análises complexas, e **Power BI** para a visualização e comunicação dos insights. Essa combinação te coloca em uma posição competitiva, pois você consegue atuar em todo o ciclo do dado, desde a extração até a apresentação para os stakeholders.

### 💪 Pontos Fortes
*   **Stack Tecnológica Equilibrada:** Você não se limita a apenas uma ferramenta, o que te dá versatilidade.
*   **Foco em Resultados:** Ao listar "Aná

In [24]:
print("\nArquivos gerados:\n")

print(
    os.listdir("output/messages")
)


Arquivos gerados:

['maria.txt', 'yago.txt', 'joão.txt']
